**Procesamiento de Imagenes   ITSE - Instituto Tecnologico de Santiago del Estero**



# Image Equalization in Image Processing

In [ ]:
import numpy as np
from  cv2 import imread
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

**Explicacion:** Se importan las librerias necesarias: `numpy` para operaciones matriciales, `cv2.imread` para leer imagenes, `cv2_imshow` para mostrar imagenes en Colab, y `matplotlib.pyplot` para graficos.

In [ ]:
def howis(img):
  print('size = ',img.shape)
  print('max  = ',np.max(img))
  print('min  = ',np.min(img))


def imhist(X):
  (N,M) = X.shape
  n = 256
  h = np.zeros((256,))
  for i in range(N):
    for j in range(M):
      x = X[i,j]
      h[x] = h[x]+1
  plt.figure(figsize=(60,9))
  plt.bar(range(n),h[0:n])
  plt.show()

**Explicacion:** Se definen dos funciones auxiliares:
- `howis(img)`: Muestra informacion de la imagen (dimensiones, valor maximo y minimo de intensidad).
- `imhist(X)`: Calcula y grafica el histograma de una imagen en escala de grises, contando cuantos pixeles hay para cada nivel de gris (0-255).

# Read the image

In [ ]:
REPO = 'https://raw.githubusercontent.com/domingomery/pr/main/clases/'
!wget {REPO}PR01_Introduccion/datos/pout.png

**Explicacion:** Se descarga la imagen `pout.png` (512x512 pixeles en escala de grises) desde un repositorio de GitHub usando el comando `wget`.

In [ ]:
X0 = imread('pout.png',0)
cv2_imshow(X0)
howis(X0)

**Explicacion:** Se lee la imagen en escala de grises (parametro `0`) y se muestra. La imagen original tiene un rango de intensidad de [74, 224], lo que significa que no utiliza todo el rango posible [0, 255], resultando en bajo contraste.

# Original Histogram

In [ ]:
imhist(X0)

**Explicacion:** Se grafica el histograma de la imagen original. Se observa que los valores de intensidad estan concentrados en un rango intermedio, confirmando el bajo contraste de la imagen.

In [ ]:
# HELP!
# CONVERSION: FLOAT (Yd) <-----> (Y) UINT8

# FLOAT is used for arithmetic operations
# UINT8 (unsigned integer of 8 bits, numbers of 8 bits, from 0 to 255) is used to display images, and some opencv functions


# CONVERSION FLOAT TO UINT8
# FLOAT (Yd) ---> UINT8 (Y)
# Y = Yd.astype(np.uint8)

# CONVERSION UINT8 TO FLOAT
# UINT8 (Y) ---> FLOAT (Yd)
# Yd = Yd.astype(float)

**Explicacion:** Referencia de conversion entre formatos: `float` se usa para operaciones aritmeticas y `uint8` (entero sin signo de 8 bits, rango 0-255) se usa para mostrar imagenes y funciones de OpenCV.

# Example 0: Spanning the histogram

linear scaling: minimum is 0 maximum is 255

In [ ]:
Xc = X0.copy()
X1 = 255 * ((Xc - Xc.min()) / (Xc.max() - Xc.min()))
X1 = X1.astype(np.uint8)
howis(X1)
cv2_imshow(X1)

**Explicacion:** Se aplica escalado lineal (stretching) para expandir el rango de intensidad de [74, 224] a [0, 255]. La formula es: `X1 = 255 * ((X - min) / (max - min))`. Esto mejora el contraste de la imagen.

In [ ]:
imhist(X1)

**Explicacion:** El histograma de la imagen con stretching ahora cubre todo el rango [0, 255], pero mantiene la misma forma que el original (solo fue escalado).

# Equalization's method 2: the output histogram is 100% uniform

In [ ]:
def hist_forceuni(img):
    img_x = img.copy()
    img_x = 255 * ((img_x - img_x.min()) / (img_x.max() - img_x.min()))

    n, m = img_x.shape
    y = np.zeros((n * m, 1), dtype=np.uint8)
    j = np.argsort(img_x.flatten())
    z = np.zeros((n * m, 1), dtype=np.uint8)

    d = int(np.fix((n * m / 256) + 0.5))
    for i in range(255):
        z[i * d:(i + 1) * d] = i * np.ones((d, 1))
    z[255 * d:n * m] = 255 * np.ones((n * m - 255 * d, 1))

    y[j] = z
    y = y.reshape(n, m)

    return y

**Explicacion:** Esta funcion implementa una ecualizacion de histograma que **fuerza** a que el histograma de salida sea 100% uniforme. El algoritmo:
1. Escala linealmente la imagen a [0, 255]
2. Ordena los pixeles por intensidad usando `argsort`
3. Calcula cuantos pixeles deberia haber por nivel de gris: `d = round(N*M / 256)`
4. Asigna valores uniformemente distribuidos (0, 1, 2, ..., 255) a los pixeles ordenados
5. Resultado: un histograma perfectamente plano

In [ ]:
X2 = hist_forceuni(X0)
howis(X2)
cv2_imshow(X2)

**Explicacion:** Se aplica la ecualizacion forzada a la imagen original `X0`. El resultado `X2` tiene un histograma completamente uniforme, lo que maximiza el contraste local de la imagen.

In [ ]:
imhist(X2)

**Explicacion:** El histograma de `X2` es perfectamente uniforme: cada nivel de gris tiene aproximadamente la misma cantidad de pixeles. Esto es lo que diferencia este metodo del metodo 1.

# Equalization's method 1: the output histogram tries to be uniform by shifting the whole bars
See implementation [from skratch](https://medium.com/hackernoon/histogram-equalization-in-python-from-scratch-ebb9c8aa3f23)

In [ ]:
from cv2 import equalizeHist
X3 = equalizeHist(X0)
cv2_imshow(X3)

**Explicacion:** Se usa la funcion `equalizeHist` de OpenCV que implementa el algoritmo estandar de ecualizacion basado en la CDF (Funcion de Distribucion Acumulada). Este metodo intenta hacer el histograma uniforme desplazando las barras, pero no logra un histograma 100% plano como el metodo 2.

In [ ]:
imhist(X3)

**Explicacion:** El histograma de `X3` es casi uniforme pero no perfecto. Se observa que el metodo 1 produce un histograma mas suave que el metodo 2 (forced uniform).

In [ ]:
Y = np.hstack((X0,X1,X2,X3))
cv2_imshow(Y)
# original, expanded, uni-forced, not-uniforced

**Explicacion:** Se muestran las 4 imagenes lado a lado para comparar:
- **X0**: Original (bajo contraste)
- **X1**: Stretching lineal (mejor contraste, misma forma de histograma)
- **X2**: Ecualizacion forzada uniforme (histograma 100% plano)
- **X3**: Ecualizacion OpenCV (histograma casi uniforme)

---
# ACTIVIDAD RESUELTA

**EJERCICIOS:**
1) Realiza una operacion aritmetica sobre la imagen para oscurecer la imagen original.
2) Realiza una operacion aritmetica para aclarar la imagen original. Ten en cuenta que la imagen no debe tener valores mayores a 255.
3) Intenta implementar una mascara para aplicar un filtro pasabajo de 3x3. Applícalo sobre la imagen ecualizada con el metodo 2 y muestra los resultados.

In [ ]:
# EJERCICIO 1: Oscurecer la imagen original
# Se multiplica cada pixel por un factor < 1 para reducir su intensidad
factor_oscurecer = 0.5
X_oscurecida = (X0.astype(float) * factor_oscurecer).astype(np.uint8)

print('=== IMAGEN OSCURECIDA ===')
howis(X_oscurecida)
cv2_imshow(X_oscurecida)

**Explicacion Ejercicio 1 - Oscurecer imagen:**

Para oscurecer la imagen, se multiplica cada pixel por un factor menor a 1 (en este caso 0.5). Esto reduce la intensidad de todos los pixeles:
- Pixel original 200 -> 200 x 0.5 = 100 (mas oscuro)
- Pixel original 100 -> 100 x 0.5 = 50 (mas oscuro)

Se usa `astype(float)` para evitar overflow al multiplicar, y luego se convierte a `uint8` para mostrar la imagen.

In [ ]:
# EJERCICIO 2: Aclarar la imagen original
# Se multiplica por un factor > 1, pero se usa np.clip para limitar a 255
factor_aclarar = 1.5
X_aclarada = np.clip(X0.astype(float) * factor_aclarar, 0, 255).astype(np.uint8)

print('=== IMAGEN ACLARADA ===')
howis(X_aclarada)
cv2_imshow(X_aclarada)

**Explicacion Ejercicio 2 - Aclarar imagen:**

Para aclarar la imagen, se multiplica cada pixel por un factor mayor a 1 (1.5). Sin embargo, esto podria generar valores mayores a 255, por lo que se usa `np.clip(array, 0, 255)` para limitar todos los valores al rango valido:
- Pixel original 150 -> 150 x 1.5 = 225 (mas claro, dentro del rango)
- Pixel original 200 -> 200 x 1.5 = 300 -> limitado a 255 (evita saturacion)

El uso de `np.clip` es fundamental para evitar valores invalidos en la imagen resultante.

In [ ]:
# EJERCICIO 3: Filtro pasabajo 3x3 sobre imagen ecualizada con metodo 2 (X2)
# Implementacion del filtro promedio 3x3 desde cero

def filtro_pasabajo_3x3(img):
    n, m = img.shape
    resultado = np.zeros((n, m), dtype=np.uint8)
    
    for i in range(1, n - 1):
        for j in range(1, m - 1):
            suma = 0
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    suma += img[i + di, j + dj]
            resultado[i, j] = int(suma / 9)
    
    return resultado

# Aplicamos el filtro a X2 (imagen ecualizada con metodo 2)
X2_filtrada = filtro_pasabajo_3x3(X2)

print('=== IMAGEN ECUALIZADA (METODO 2) CON FILTRO PASABAJO 3x3 ===')
howis(X2_filtrada)
cv2_imshow(X2_filtrada)

**Explicacion Ejercicio 3 - Filtro pasabajo 3x3:**

El filtro pasabajo (tambien llamado filtro de promedio o blur) suaviza la imagen promediando cada pixel con sus 8 vecinos mas cercanos (ventana 3x3).

**Mascara utilizada:**
```
1/9 x | 1  1  1 |
       | 1  1  1 |
       | 1  1  1 |
```

**Proceso:**
1. Se coloca la mascara sobre cada pixel de la imagen
2. Se multiplica cada valor de la mascara por el pixel correspondiente
3. Se suman todos los resultados y se divide por 9 (promedio)
4. El resultado reemplaza al pixel central

**Efecto:** Reduce el ruido y suaviza los detalles/bordes de la imagen. Se aplico sobre `X2` (imagen ecualizada con histograma uniforme) para observar el efecto de suavizado sobre una imagen de alto contraste.

In [ ]:
# Comparacion de resultados de la actividad
Y_actividad = np.hstack((X0, X_oscurecida, X_aclarada, X2, X2_filtrada))
cv2_imshow(Y_actividad)
# Original | Oscurecida | Aclarada | Ecualizada Metodo 2 | Ecualizada + Filtro

**Explicacion comparacion final:**

Se muestran las 5 imagenes lado a lado:
- **Original**: Bajo contraste, rango [74, 224]
- **Oscurecida**: Misma imagen con intensidad reducida (factor 0.5)
- **Aclarada**: Misma imagen con intensidad aumentada (factor 1.5, limitada a 255)
- **Ecualizada Metodo 2**: Histograma 100% uniforme, maximo contraste
- **Ecualizada + Filtro**: Imagen suavizada, se perdieron detalles y bordes

**Bibliografia**  
D. Mery. Fundamentos de Procesamiento de Imagenes Departamento de Ciencia de la Computacion. Universidad Catolica de Chile, 2024.